<a href="https://colab.research.google.com/github/Fu-Pei-Yin/Deep-Generative-Mode/blob/week9/LLM_%E5%BE%AE%E8%AA%BF%EF%BC%9A%E6%83%85%E7%B7%92%E5%88%86%E9%A1%9E%E8%88%87%E6%86%82%E9%AC%B1%E7%97%87%E9%A2%A8%E9%9A%AA%E7%9B%A3%E6%B8%AC%EF%BC%88LoRA_Zero_shot_Few_shot%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
HW9 - LLM 微調：情緒分類與憂鬱症風險監測
使用 LoRA / Zero-shot / Few-shot 方法
"""

# ============================================================================
# 1. 安裝必要套件
# ============================================================================
"""
在 Colab 中執行以下指令：
!pip install datasets transformers accelerate peft bitsandbytes
!pip install scikit-learn matplotlib seaborn pandas numpy
!pip install torch torchvision torchaudio
"""

# ============================================================================
# 2. 匯入套件
# ============================================================================
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

# 設定隨機種子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# ============================================================================
# 3. 載入情緒資料集
# ============================================================================
print("載入 Emotion Dataset...")
dataset = load_dataset("dair-ai/emotion")

# 查看資料集結構
print("\n資料集資訊:")
print(dataset)
print("\n訓練集範例:")
print(dataset['train'][0])

# 情緒標籤對應
emotion_labels = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

# ============================================================================
# 4. 風險映射規則 (Risk Mapping)
# ============================================================================
def emotion_to_risk(emotion_label):
    """
    將情緒標籤轉換為風險等級
    joy/love/surprise → 0 = low_risk
    anger/fear → 1 = mid_risk
    sadness → 2 = high_risk
    """
    emotion_name = emotion_labels[emotion_label]

    if emotion_name in ['joy', 'love', 'surprise']:
        return 0  # low_risk
    elif emotion_name in ['anger', 'fear']:
        return 1  # mid_risk
    elif emotion_name == 'sadness':
        return 2  # high_risk
    else:
        return 0

risk_labels = {
    0: 'low_risk',
    1: 'mid_risk',
    2: 'high_risk'
}

# 為資料集添加風險標籤
def add_risk_labels(examples):
    examples['risk_label'] = [emotion_to_risk(label) for label in examples['label']]
    return examples

dataset = dataset.map(add_risk_labels, batched=True)

print("\n添加風險標籤後的範例:")
for i in range(3):
    example = dataset['train'][i]
    print(f"文本: {example['text']}")
    print(f"情緒: {emotion_labels[example['label']]}")
    print(f"風險: {risk_labels[example['risk_label']]}\n")

# ============================================================================
# 5. 模型與 Tokenizer 設定
# ============================================================================
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 可替換為其他模型
print(f"\n載入模型: {MODEL_NAME}")

# 量化配置（節省記憶體）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 載入 tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 載入模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# ============================================================================
# 6. Prompt 設計
# ============================================================================
def create_prompt(text, task="emotion", few_shot_examples=None):
    """
    創建 prompt 用於情緒分類或風險評估
    """
    if task == "emotion":
        instruction = "Classify the emotion of the following text. Choose from: sadness, joy, love, anger, fear, surprise."
        response_format = "Emotion:"
    else:  # risk
        instruction = "Assess the depression risk level of the following text. Choose from: low_risk, mid_risk, high_risk."
        response_format = "Risk:"

    prompt = f"<|system|>\nYou are an emotion analysis expert.</s>\n<|user|>\n{instruction}\n\n"

    # Few-shot examples
    if few_shot_examples:
        for example in few_shot_examples:
            prompt += f"Text: {example['text']}\n{response_format} {example['label']}\n\n"

    prompt += f"Text: {text}\n{response_format}"

    return prompt

# ============================================================================
# 7. Zero-shot 推論
# ============================================================================
def zero_shot_inference(texts, task="emotion", max_samples=100):
    """
    Zero-shot 推論
    """
    print(f"\n執行 Zero-shot 推論 ({task})...")
    predictions = []

    for i, text in enumerate(texts[:max_samples]):
        if i % 20 == 0:
            print(f"處理進度: {i}/{max_samples}")

        prompt = create_prompt(text, task=task)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # 簡單解析（實際應用中需要更複雜的解析邏輯）
        predictions.append(response)

    return predictions

# ============================================================================
# 8. Few-shot 推論
# ============================================================================
def few_shot_inference(texts, examples, task="emotion", max_samples=100):
    """
    Few-shot 推論
    """
    print(f"\n執行 Few-shot 推論 ({task})...")
    predictions = []

    for i, text in enumerate(texts[:max_samples]):
        if i % 20 == 0:
            print(f"處理進度: {i}/{max_samples}")

        prompt = create_prompt(text, task=task, few_shot_examples=examples)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        predictions.append(response)

    return predictions

# 準備 few-shot examples
few_shot_examples = [
    {"text": "i feel so happy today", "label": "joy"},
    {"text": "this makes me very angry", "label": "anger"},
    {"text": "i am feeling so sad and hopeless", "label": "sadness"},
]

# ============================================================================
# 9. LoRA 微調設定
# ============================================================================
# 準備模型進行訓練
model = prepare_model_for_kbit_training(model)

# LoRA 配置
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 應用 LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================================
# 10. 資料預處理
# ============================================================================
def preprocess_function(examples):
    """
    預處理訓練資料
    """
    prompts = []
    for text, label in zip(examples['text'], examples['label']):
        emotion = emotion_labels[label]
        prompt = f"<|system|>\nYou are an emotion analysis expert.</s>\n<|user|>\nClassify the emotion: {text}</s>\n<|assistant|>\n{emotion}</s>"
        prompts.append(prompt)

    model_inputs = tokenizer(
        prompts,
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

# 處理資料集
print("\n預處理資料集...")
tokenized_train = dataset['train'].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['train'].column_names
)
tokenized_val = dataset['validation'].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset['validation'].column_names
)

# ============================================================================
# 11. 訓練參數設定
# ============================================================================
training_args = TrainingArguments(
    output_dir="./emotion-lora-model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",  # 已更新：從 evaluation_strategy 改為 eval_strategy
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    warmup_steps=100,
    optim="paged_adamw_8bit"
)

# ============================================================================
# 12. 訓練模型
# ============================================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print("\n開始訓練...")
trainer.train()  # 取消註解以執行訓練

# ============================================================================
# 13. 評估函數
# ============================================================================
def evaluate_model(y_true, y_pred, num_classes=6):
    """
    評估模型效能
    """
    # F1 Score
    f1_micro = f1_score(y_true, y_pred, average='micro')
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')

    print(f"F1 Score (Micro): {f1_micro:.4f}")
    print(f"F1 Score (Macro): {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=list(emotion_labels.values())))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=emotion_labels.values(),
                yticklabels=emotion_labels.values())
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'confusion_matrix': cm
    }

# ============================================================================
# 14. 風險監測視覺化
# ============================================================================
def visualize_risk_monitoring(risk_probs, window_size=50):
    """
    視覺化風險監測
    1. 高風險走勢圖
    2. 高風險濃度熱圖
    """
    # 1. 高風險走勢圖
    plt.figure(figsize=(15, 5))
    plt.plot(risk_probs, alpha=0.6, linewidth=1)
    plt.axhline(y=0.5, color='r', linestyle='--', label='High Risk Threshold (0.5)')
    plt.xlabel('Sample Index')
    plt.ylabel('P(high_risk)')
    plt.title('High Risk Probability Trend')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('risk_trend.png', dpi=300, bbox_inches='tight')
    plt.show()

    # 2. 高風險濃度熱圖 (Rolling Window)
    rolling_mean = pd.Series(risk_probs).rolling(window=window_size, min_periods=1).mean()

    # 重塑為 2D 以便繪製熱圖
    n_samples = len(rolling_mean)
    n_cols = 50
    n_rows = (n_samples + n_cols - 1) // n_cols

    # 填充到完整矩陣
    padded_data = np.pad(rolling_mean, (0, n_rows * n_cols - n_samples),
                         mode='constant', constant_values=np.nan)
    heatmap_data = padded_data.reshape(n_rows, n_cols)

    plt.figure(figsize=(15, 8))
    sns.heatmap(heatmap_data, cmap='YlOrRd', cbar_kws={'label': 'Risk Level'},
                vmin=0, vmax=1, linewidths=0)
    plt.title(f'High Risk Concentration Heatmap (Rolling Window = {window_size})')
    plt.xlabel('Sample Index (within row)')
    plt.ylabel('Row')
    plt.tight_layout()
    plt.savefig('risk_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

# ============================================================================
# 15. 主要執行流程
# ============================================================================
def main():
    """
    主要執行流程
    """
    print("="*80)
    print("情緒分類與憂鬱症風險監測系統")
    print("="*80)

    # 取得測試資料
    test_texts = dataset['test']['text']
    test_labels = dataset['test']['label']
    test_risks = dataset['test']['risk_label']

    # 示範評估（使用隨機預測作為範例）
    print("\n生成示範預測...")
    dummy_predictions = np.random.randint(0, 6, size=len(test_labels))
    dummy_risk_probs = np.random.rand(len(test_risks))

    # 評估情緒分類
    print("\n情緒分類評估結果:")
    evaluate_model(test_labels, dummy_predictions)

    # 視覺化風險監測
    print("\n生成風險監測視覺化...")
    visualize_risk_monitoring(dummy_risk_probs)

    print("\n完成！請查看生成的圖表和評估結果。")
    print("\n注意：以上使用隨機預測作為示範。")
    print("取消 trainer.train() 的註解以執行實際訓練。")

# 執行主程式
if __name__ == "__main__":
    main()

# ============================================================================
# 16. 結果比較表格
# ============================================================================
def create_comparison_table():
    """
    創建方法比較表格
    """
    results = {
        'Method': ['Zero-shot', 'Few-shot', 'LoRA Fine-tuned'],
        'F1 (Macro)': [0.0, 0.0, 0.0],  # 填入實際結果
        'F1 (Weighted)': [0.0, 0.0, 0.0],
        'Training Time': ['0 min', '0 min', '~30 min'],
        'Parameters Updated': ['0', '0', '~1M'],
    }

    df = pd.DataFrame(results)
    print("\n方法比較:")
    print(df.to_string(index=False))
    return df

create_comparison_table()

print("\n"+"="*80)
print("程式碼架構完成！")
print("="*80)
print("\n使用說明:")
print("1. 在 Colab 中執行此程式")
print("2. 確保已安裝所有必要套件")
print("3. 取消 trainer.train() 的註解以執行訓練")
print("4. 根據需求調整模型、參數和視覺化設定")
print("5. 完整執行後會生成評估指標和視覺化圖表")

載入 Emotion Dataset...

資料集資訊:
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

訓練集範例:
{'text': 'i didnt feel humiliated', 'label': 0}

添加風險標籤後的範例:
文本: i didnt feel humiliated
情緒: sadness
風險: high_risk

文本: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
情緒: sadness
風險: high_risk

文本: im grabbing a minute to post i feel greedy wrong
情緒: anger
風險: mid_risk


載入模型: TinyLlama/TinyLlama-1.1B-Chat-v1.0
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079

預處理資料集...


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


開始訓練...


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
